# 03B — Pemodelan BiLSTM pada Data Baru (v2): 3 Skenario Simulasi Ketimpangan

Notebook ini menguji ketahanan model BiLSTM terhadap 3 rasio ketimpangan data latih buatan:
1. **Skenario 1:1:1** (Seimbang Sempurna: 33.3% Neg, 33.3% Net, 33.3% Pos)
2. **Skenario 6:3:1** (Ketimpangan Moderat: 60% Neg, 10% Net, 30% Pos)
3. **Skenario 8:1:1** (Ketimpangan Ekstrem / Long-tail: 80% Neg, 10% Net, 10% Pos)

Setiap skenario dievaluasi pada **Test Set Empiris Terkunci ($n = 1.730$)** dengan varian **Baseline Natural** dan **Random Oversampling (ROS)** untuk mengukur pemulihan performa terhadap *Majority Collapse*.


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file secara rekursif di /kaggle/input (Kaggle) atau kandidat lokal."""
    # 1. Rekursif cari di /kaggle/input (menangani semua variasi mount Kaggle)
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Kandidat lokal workstation
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/raw/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"kamus/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/raw/{filename}"),
        Path(f"../kamus/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename


In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score
from imblearn.over_sampling import RandomOverSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Test Set Empiris Terkunci
csv_path = resolve_path('banjir_processed_v2.csv')
df_full = pd.read_csv(csv_path)
col_text = 'processed_text_v2'
col_label = 'label'

_, test_df = train_test_split(df_full, test_size=0.20, stratify=df_full[col_label], random_state=SEED)
X_test_raw = test_df[col_text].astype(str).values
y_test = test_df[col_label].values

print(f'Test set terkunci: {len(test_df)} sampel')
print('Distribusi test label:', pd.Series(y_test).value_counts().sort_index().to_dict())


In [ ]:
def build_bilstm_model(vocab_size=10000, embedding_dim=128, units=64, dropout=0.3, max_len=50, lr=0.0001):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
        Bidirectional(LSTM(units)),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Arsitektur BiLSTM siap.')


In [ ]:
# Eksekusi Simulasi BiLSTM: Baseline vs ROS pada 3 Skenario
scenarios = ['111', '631', '811']
results = []

for sc in scenarios:
    sc_file = f'scenario_{sc}.csv'
    sc_path = resolve_path(sc_file)
    print('=' * 65)
    print(f'MEMPROSES SKENARIO {sc}: {sc_path}')
    print('=' * 65)
    
    df_sc = pd.read_csv(sc_path)
    tr_sub, val_sub = train_test_split(df_sc, test_size=0.10, stratify=df_sc[col_label], random_state=SEED)
    
    tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
    tokenizer.fit_on_texts(tr_sub[col_text].astype(str))
    
    X_tr = pad_sequences(tokenizer.texts_to_sequences(tr_sub[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
    X_va = pad_sequences(tokenizer.texts_to_sequences(val_sub[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
    X_te = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=50, padding='post', truncating='post')
    y_tr = tr_sub[col_label].values
    y_va = val_sub[col_label].values
    
    # 1. Baseline Run (Natural)
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    m_base = build_bilstm_model()
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m_base.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=20, batch_size=16, callbacks=[es], verbose=0)
    pred_base = np.argmax(m_base.predict(X_te, verbose=0), axis=1)
    f1_base = f1_score(y_test, pred_base, average='macro', zero_division=0)
    acc_base = accuracy_score(y_test, pred_base)
    rec_net_base = recall_score(y_test, pred_base, average=None, zero_division=0)[1]
    
    # 2. ROS Run (Random Oversampling)
    ros = RandomOverSampler(random_state=SEED)
    X_tr_ros, y_tr_ros = ros.fit_resample(X_tr, y_tr)
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    m_ros = build_bilstm_model()
    es_ros = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m_ros.fit(X_tr_ros, y_tr_ros, validation_data=(X_va, y_va), epochs=20, batch_size=16, callbacks=[es_ros], verbose=0)
    pred_ros = np.argmax(m_ros.predict(X_te, verbose=0), axis=1)
    f1_ros = f1_score(y_test, pred_ros, average='macro', zero_division=0)
    acc_ros = accuracy_score(y_test, pred_ros)
    rec_net_ros = recall_score(y_test, pred_ros, average=None, zero_division=0)[1]
    
    delta_f1 = (f1_ros - f1_base) * 100
    print(f'Skenario {sc} | Baseline F1: {f1_base*100:.2f}% | ROS F1: {f1_ros*100:.2f}% | Delta: {delta_f1:+.2f} pp')
    
    results.append({
        'Skenario': sc,
        'Baseline_Acc': acc_base,
        'Baseline_F1': f1_base,
        'Baseline_RecNet': rec_net_base,
        'ROS_Acc': acc_ros,
        'ROS_F1': f1_ros,
        'ROS_RecNet': rec_net_ros,
        'Delta_F1_pp': delta_f1
    })


In [ ]:
# Rangkuman & Ekspor Hasil Simulasi BiLSTM
df_res = pd.DataFrame(results)
print('=' * 75)
print('TABEL HASIL SIMULASI BiLSTM: BASELINE VS ROS PEMULIH')
print('=' * 75)
for _, r in df_res.iterrows():
    print(f"Skenario {r['Skenario']}: Baseline F1 = {r['Baseline_F1']*100:.2f}%, ROS F1 = {r['ROS_F1']*100:.2f}% (Delta: {r['Delta_F1_pp']:+.2f} pp)")

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('Output/simulated')
out_dir.mkdir(parents=True, exist_ok=True)
df_res.to_csv(out_dir / 'results_bilstm_simulasi.csv', index=False)
print(f'Hasil disimpan di {out_dir / "results_bilstm_simulasi.csv"}')
